# Technique 1: Case 1


Run all cells to obtain every configuration. Convergence and `%%timeit` use separate cells. To repeat timings, rerun each method’s setup and timing cells, leaving its convergence cell unchanged. All timing cells run serially; integrators are constructed beforehand.


In [1]:
import os
import sys
import copy
import importlib
import json
import platform
from pathlib import Path

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import numpy as np
import scipy
import heyoka as hy
from IPython.display import display, Markdown

hy.set_nthreads(1)


In [2]:
# =========================
# Timing Settings
# =========================
from collections import Counter

cpu_samples = []
fast_cpus = []
if sys.platform == 'darwin':
    import ctypes
    import plistlib
    import subprocess
    tree = plistlib.loads(subprocess.check_output(
        ['/usr/sbin/ioreg', '-a', '-l', '-p', 'IODeviceTree', '-n', 'cpus', '-r']))
    pending = list(tree)
    while pending:
        node = pending.pop()
        if node.get('cluster-type') == b'P\x00' and 'cpu-id' in node:
            fast_cpus.append(int.from_bytes(node['cpu-id'], 'little'))
        pending.extend(node.get('IORegistryEntryChildren', []))
    lib = ctypes.CDLL('/usr/lib/libSystem.B.dylib')
    lib.pthread_set_qos_class_self_np.argtypes = [ctypes.c_uint, ctypes.c_int]
    lib.pthread_cpu_number_np.argtypes = [ctypes.POINTER(ctypes.c_size_t)]

def set_timing_priority():
    if sys.platform == 'darwin':
        assert lib.pthread_set_qos_class_self_np(0x21, 0) == 0

def sample_cpu():
    if sys.platform == 'darwin':
        cpu = ctypes.c_size_t()
        assert lib.pthread_cpu_number_np(ctypes.byref(cpu)) == 0
        cpu_samples.append(cpu.value)

environment = dict(python=sys.version, executable=sys.executable,
                   platform=platform.platform(), machine=platform.machine(),
                   numpy=np.__version__, scipy=scipy.__version__, heyoka=hy.__version__,
                   threads=1, fast_cpus=sorted(fast_cpus), strict_cpu_pinning=False)
print(environment)

records_by_method = {}
results_by_method = {}

def save_trial(i, result, accepted, nfev):
    records.append(dict(trial=int(i), accepted=bool(accepted), nfev=int(nfev),
        status=int(result.status) if result is not None else None,
        residual_norm=float(np.linalg.norm(result.fun)) if result is not None else None,
        x=result.x.tolist() if result is not None else None))


{'python': '3.12.14 (main, Sep  3 2026, 03:46:19) [Clang 21.1.8 ]', 'executable': '/Users/zhong/Documents/Codex/miniforge3/envs/gtoc13_heyoka7121_paper/bin/python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'machine': 'arm64', 'numpy': '2.5.3', 'scipy': '1.18.0', 'heyoka': '7.12.1', 'threads': 1, 'fast_cpus': [12, 13, 14, 15, 16, 17], 'strict_cpu_pinning': False}


In [3]:
import numpy as np
import heyoka as hy
AU2m = 149597870700
day2s = 86400.0
tof = 3534.0 * day2s
m0 = 4000.0
Tmax = 0.32
Isp = 3000
mu = 132712440018 * 1000000000.0
g0 = 9.8065
mee0 = [149554268256.46957, -0.003766786954568226, 0.016286832739416963, -7.702048962437313e-06, 6.183156622292267e-07, 1.5955219194574601]
meef = [232433083985.70743, 0.1530290696088389, -0.5199481742007113, 0.01618310223871937, 0.11813952745106715, 33.763869726200596]
LUnit = AU2m
TUnit = np.sqrt(LUnit * LUnit * LUnit / mu)
MUnit = m0
VUnit = LUnit / TUnit
AUnit = VUnit / TUnit
x0_nu = mee0
xf_nu = meef
t0_nu = 0.0
tf_nu = t0_nu + tof
m0_nu = m0
mu_nu = mu
Tmax_nu = Tmax
Isp_nu = Isp
g0_nu = g0
x0_nu[0] = x0_nu[0] / LUnit
xf_nu[0] = xf_nu[0] / LUnit
t0_nu = t0_nu / TUnit
tf_nu = tf_nu / TUnit
m0_nu = m0_nu / MUnit
mu_nu = mu_nu * TUnit * TUnit / LUnit / LUnit / LUnit
Tmax_nu = Tmax_nu / MUnit / AUnit
Isp_nu = Isp_nu / TUnit
g0_nu = g0_nu / AUnit
c1 = Tmax_nu
c2 = Isp_nu * g0_nu

import time
from scipy.optimize import root
repeat_num_minpack = 5000
minpack_tol = 1e-10
minpack_xtol = 1e-10
minpack_maxfev = 50000
minpack_factor = 0.1
minpack_diag = [1.0] * 3 + [1.0] * 3 + [1.0] * 1
minpack_eps = 0.0
minpack_error = 1e-07
np.set_printoptions(precision=16, suppress=True)


import twobody_MEE_hamilton_log as model
state_model = model
TEST_ID = "MEE_CHAINRULE_FULL_STM_20260913"
FF_STM_LAYOUT = "fused"
MODEL_ID = model.MODEL_ID
# Residual and STM integrators share the same simplified MEE equations.
# Sensitivities use the exact hand-derived chain rule. FF retains all 196 entries.
# Fresh mutable copies are owned by this Python process.
ta = state_model.get_integrator(compact=False)
ta_var = model.get_integrator(7, compact=False)
ta_var_ff = model.get_integrator(14, compact=False)
ta_b2 = state_model.get_integrator(batch=2, compact=False)
ta_var_fb_b2 = model.get_integrator(7, batch=2, compact=False)
epsf = 1e-5
lambda0 = 1.0
for integrator in [ta, ta_var, ta_var_ff]:
    integrator.pars[:] = [mu_nu, c1, c2, epsf, lambda0]
for integrator in [ta_b2, ta_var_fb_b2]:
    integrator.pars[:] = np.tile(np.array([mu_nu, c1, c2, epsf, lambda0])[:, None], (1, 2))


# Always build the full original pool before selecting or partitioning trials.
repeat_num_minpack = 5000
RANDOM_SEED = 322
FB_AUX_SEED = 1322


In [4]:
def build_common_lam0_pool(n_trials, seed=RANDOM_SEED):
    """
    Build the common 7D initial co-state pool used by method1-5.

    The per-trial draw order intentionally reproduces the original
    method1/method2 logic:
        7 draws mapped to (-1, 1), followed by a separate draw for lambda_m.
    """
    rng = np.random.RandomState(seed)
    pool = np.empty((int(n_trials), 7), dtype=float)
    for i in range(int(n_trials)):
        pool[i] = 2.0 * rng.random_sample(7) - 1.0
        pool[i, 6] = rng.random_sample()
    return pool


In [5]:
def build_fb_z0_pool(lam0_pool, m0=1.0, aux_seed=FB_AUX_SEED):
    """
    Build full 14D initial vectors for method4/5:
        z0 = [lam0(7), mf0(1), lam_rf0(3), lam_vf0(3)].

    The first seven entries are copied from the shared pool. The additional
    forward-backward variables come from an independent RNG stream.
    """
    lam0_pool = np.asarray(lam0_pool, dtype=float)
    if lam0_pool.ndim != 2 or lam0_pool.shape[1] != 7:
        raise ValueError('lam0_pool must have shape (n_trials, 7).')
    n_trials = lam0_pool.shape[0]
    rng_aux = np.random.RandomState(aux_seed)
    mf0 = np.full((n_trials, 1), 0.9 * float(m0), dtype=float)
    lrf0 = 2.0 * rng_aux.random_sample((n_trials, 3)) - 1.0
    lvf0 = 2.0 * rng_aux.random_sample((n_trials, 3)) - 1.0
    z0_pool = np.hstack([lam0_pool.copy(), mf0, lrf0, lvf0])
    assert np.array_equal(z0_pool[:, :7], lam0_pool)
    return z0_pool


In [6]:
lam0_pool = build_common_lam0_pool(repeat_num_minpack, seed=RANDOM_SEED)


In [7]:
fb_z0_pool = build_fb_z0_pool(lam0_pool, m0=m0_nu, aux_seed=FB_AUX_SEED)


In [8]:
def shooting(lam, eps=1e-05, lambda0=1.0, out=False, Time_out=None):
    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam[:]
    ta.propagate_until(tf_nu)
    retval = [ta.state[0] - xf_nu[0], ta.state[1] - xf_nu[1], ta.state[2] - xf_nu[2], ta.state[3] - xf_nu[3], ta.state[4] - xf_nu[4], ta.state[5] - xf_nu[5], ta.state[13]]
    if out:
        print('  Final Mass:', ta.state[6] * MUnit)
    if Time_out is not None:
        test = Time_out
    return retval


In [9]:
def stm_single(lam, eps=1e-5, lambda0=1., tf=tf_nu):
    """Seven initial-costate directions, with lambda0 held fixed."""
    ta_var.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta_var.time = t0_nu
    ta_var.state[:14] = np.r_[x0_nu, m0_nu, lam]
    initial = np.zeros((14, 7))
    initial[7:14, :] = np.eye(7)
    ta_var.state[14:] = initial.ravel()
    ta_var.propagate_until(tf)
    return ta_var.state[14:].reshape(14, 7).copy()


In [10]:
def stm_segment_selected(Xk_14, l0, t_start, t_end, active_cols=None):
    """Return the complete requested sensitivities in a 14 by 14 matrix.

    FF always computes columns 0 through 13, including on the first arc.
    Splitting the columns changes execution and adaptive step control only.
    State returns follow the final block; shooting residuals use the separate
    unchanged 1e-12 physical-state integrator.
    """
    rows = np.arange(14) if active_cols is None else np.asarray(list(active_cols), dtype=int)
    if len(rows) not in (7,14) or len(np.unique(rows))!=len(rows) or np.any((rows<0)|(rows>=14)):
        raise ValueError("Use seven selected rows or all fourteen rows")
    # A full STM has canonical output columns regardless of requested row order.
    if len(rows)==14 and active_cols is not None:
        rows = np.arange(14)
    initial = np.zeros((14,len(rows)))
    initial[rows,np.arange(len(rows))] = 1.
    parameters = [mu_nu,c1,c2,epsf,l0]
    phi = np.zeros((14,14))
    if len(rows)==14 and FF_STM_LAYOUT=="blocks_simd":
        t = ta_var_fb_b2
        t.pars[:] = np.array(parameters)[:,None]
        t.set_time([float(t_start)]*2)
        t.state[:14,:] = np.asarray(Xk_14)[:,None]
        t.state[14:,:] = np.stack([initial[:,:7],initial[:,7:]],axis=-1).reshape(98,2)
        t.propagate_until([float(t_end)]*2)
        blocks = t.state[14:,:].reshape(14,7,2)
        phi[:,:7],phi[:,7:] = blocks[:,:,0],blocks[:,:,1]
        end = t.state[:14,1].copy()
    else:
        chunks = [(0,14)] if len(rows)==14 and FF_STM_LAYOUT=="fused" else [(j,j+7) for j in range(0,len(rows),7)]
        for first,last in chunks:
            t = ta_var_ff if last-first==14 else ta_var
            t.pars[:] = parameters
            t.time = float(t_start)
            t.state[:14] = Xk_14
            t.state[14:] = initial[:,first:last].ravel()
            t.propagate_until(float(t_end))
            phi[:,rows[first:last]] = t.state[14:].reshape(14,last-first)
            end = t.state[:14].copy()
    return end[:7],end[7:14],phi


In [11]:
def shooting_jac(lam, eps=1e-05, lambda0=1.0):
    """
    Return the shooting Jacobian J with shape (7,7),
    w.r.t. the 7 initial co-states.
    """
    Phi_tf = stm_single(lam, eps, lambda0)
    J = np.zeros((7, 7))
    J[0:6, :] = Phi_tf[0:6, 0:7]
    J[6, :] = Phi_tf[13, 0:7]
    return J


In [12]:
def _ms_split_time(N, tf_val=None, t0_val=t0_nu):
    """
    Divide [t0_val, tf_val] into N equal time segments where N >= 2.
    """
    assert N >= 2, 'Number of segments N must be >= 2'
    T = float(tf_nu if tf_val is None else tf_val)
    dt = (T - t0_val) / N
    t_nodes = np.linspace(t0_val, T, N + 1)
    return (t_nodes, dt)


In [13]:
def _ms_unpack_z(z, N):
    """
    Variable order:
        z = [Lambda_0, ..., Lambda_{N-1}, X_1, ..., X_{N-1}]
    where Lambda_k and X_k are 7-dimensional.
    In the current problem, lambda0 is fixed to 1.0.
    """
    z = np.asarray(z, dtype=float)
    n_lam = 7 * N
    n_X = 7 * (N - 1)
    lam_flat = z[:n_lam]
    X_flat = z[n_lam:n_lam + n_X]
    l0 = lambda0
    lam_nodes = lam_flat.reshape((N, 7))
    X_nodes = X_flat.reshape((N - 1, 7))
    return (lam_nodes, X_nodes, float(l0))


In [14]:
def _ms_pack_z(lam_nodes, X_nodes):
    """
    Pack [Lambda_0, ..., Lambda_{N-1}, X_1, ..., X_{N-1}] into one vector.
    """
    return np.hstack([lam_nodes.ravel(), X_nodes.ravel()])


In [15]:
def ms_jacobian(z, N):
    """
    Return J = ∂res/∂z ∈ R^{(14N-7)×(14N-7)} as a numpy array.

    Variable order:
        z = [Lambda_0, ..., Lambda_{N-1}, X_1, ..., X_{N-1}]
    where each Lambda_k and X_k is 7-dimensional, and lambda0 is fixed.

    Row block order:
        continuity per segment: 14*(N-1)
        terminal conditions: 7
    """
    lam_nodes, X_nodes, l0 = _ms_unpack_z(z, N)
    t_nodes, _ = _ms_split_time(N, tf_nu)
    n_var = 14 * N - 7
    J = np.zeros((n_var, n_var), dtype=float)

    def cidx_lam(k):
        s = 7 * k
        return slice(s, s + 7)

    def cidx_X(k):
        base = 7 * N
        s = base + 7 * (k - 1)
        return slice(s, s + 7)
    row = 0
    X0 = np.array(list(x0_nu) + [m0_nu], dtype=float)
    Xk_14 = np.r_[X0, lam_nodes[0]]
    X_end, L_end, Phi = stm_segment_selected(Xk_14, l0, t_nodes[0], t_nodes[1])
    A = Phi[:, :14]
    J[row:row + 14, cidx_lam(0)] += A[:, 7:14]
    J[row:row + 7, cidx_X(1)] -= np.eye(7)
    J[row + 7:row + 14, cidx_lam(1)] -= np.eye(7)
    row += 14
    for k in range(1, N - 1):
        Xk_14 = np.r_[X_nodes[k - 1], lam_nodes[k]]
        X_end, L_end, Phi = stm_segment_selected(Xk_14, l0, t_nodes[k], t_nodes[k + 1])
        A = Phi[:, :14]
        J[row:row + 14, cidx_X(k)] += A[:, 0:7]
        J[row:row + 14, cidx_lam(k)] += A[:, 7:14]
        J[row:row + 7, cidx_X(k + 1)] -= np.eye(7)
        J[row + 7:row + 14, cidx_lam(k + 1)] -= np.eye(7)
        row += 14
    Xk_14 = np.r_[X_nodes[N - 2], lam_nodes[N - 1]]
    X_end, L_end, Phi = stm_segment_selected(Xk_14, l0, t_nodes[N - 1], t_nodes[N])
    A = Phi[:, :14]
    rows_sel = [0, 1, 2, 3, 4, 5, 13]
    A_sel = A[rows_sel, :]
    J[row:row + 7, cidx_X(N - 1)] += A_sel[:, 0:7]
    J[row:row + 7, cidx_lam(N - 1)] += A_sel[:, 7:14]
    row += 7
    return J


In [16]:
def ms_residual(z, N):
    """
    Return residual of length 14N-7:
      - Continuity: 14 per segment, total 14*(N-1)
      - Terminal: 7 ([x,y,z,vx,vy,vz, lm(tf)])
    """
    lam_nodes, X_nodes, l0 = _ms_unpack_z(z, N)
    t_nodes, _ = _ms_split_time(N, tf_nu)
    X0 = np.array(list(x0_nu) + [m0_nu], dtype=float)
    res = []
    ta.time = float(t_nodes[0])
    ta.state[:7] = X0.tolist()
    ta.state[7:14] = np.array(lam_nodes[0], dtype=float).tolist()
    ta.pars[4] = float(l0)
    ta.propagate_until(float(t_nodes[1]))
    res.extend((np.array(ta.state[:7], dtype=float) - np.array(X_nodes[0], dtype=float)).tolist())
    res.extend((np.array(ta.state[7:14], dtype=float) - np.array(lam_nodes[1], dtype=float)).tolist())
    for k in range(1, N - 1):
        ta.time = float(t_nodes[k])
        ta.state[:7] = np.array(X_nodes[k - 1], dtype=float).tolist()
        ta.state[7:14] = np.array(lam_nodes[k], dtype=float).tolist()
        ta.pars[4] = float(l0)
        ta.propagate_until(float(t_nodes[k + 1]))
        res.extend((np.array(ta.state[:7], dtype=float) - np.array(X_nodes[k], dtype=float)).tolist())
        res.extend((np.array(ta.state[7:14], dtype=float) - np.array(lam_nodes[k + 1], dtype=float)).tolist())
    ta.time = float(t_nodes[N - 1])
    ta.state[:7] = np.array(X_nodes[N - 2], dtype=float).tolist()
    ta.state[7:14] = np.array(lam_nodes[N - 1], dtype=float).tolist()
    ta.pars[4] = float(l0)
    ta.propagate_until(float(t_nodes[N]))
    X_end = np.array(ta.state[:7], dtype=float)
    L_end = np.array(ta.state[7:14], dtype=float)
    res.extend([X_end[0] - xf_nu[0], X_end[1] - xf_nu[1], X_end[2] - xf_nu[2], X_end[3] - xf_nu[3], X_end[4] - xf_nu[4], X_end[5] - xf_nu[5], L_end[6]])
    return np.asarray(res, dtype=float)


In [17]:
def ms_build_initial_guess(N, lam0_guess=None, rng=None, max_attempts=100):
    """
    Build an initial guess for forward-forward multiple shooting.

    For a supplied lam0_guess, exactly that trial is used once. If its
    propagated node guess is invalid, RuntimeError is raised; the function
    does not silently draw a replacement, because that would break paired
    comparisons across methods.

    If lam0_guess is None, random retry remains available for non-benchmark use.
    """
    assert N >= 2
    X0 = np.array(list(x0_nu) + [m0_nu], dtype=float)
    t_nodes, _ = _ms_split_time(N, tf_nu)
    fixed_lam0 = lam0_guess is not None
    if fixed_lam0:
        lam0_fixed = np.asarray(lam0_guess, dtype=float)
        if lam0_fixed.shape != (7,):
            raise ValueError('lam0_guess must be a 7-dimensional vector.')
        attempts = 1
    else:
        if rng is None:
            rng = np.random
        attempts = int(max_attempts)
        if attempts < 1:
            raise ValueError('max_attempts must be >= 1.')
    for _ in range(attempts):
        if fixed_lam0:
            lam0 = lam0_fixed.copy()
        else:
            lam0 = 2.0 * rng.random(7) - 1.0
            lam0[-1] = rng.random()
        lam_nodes = np.zeros((N, 7), dtype=float)
        X_nodes = np.zeros((N - 1, 7), dtype=float)
        ta.time = float(t_nodes[0])
        ta.state[:7] = X0.tolist()
        ta.state[7:14] = lam0.tolist()
        ta.pars[4] = float(lambda0)
        lam_nodes[0] = lam0
        valid = True
        for k in range(1, N + 1):
            ta.propagate_until(float(t_nodes[k]))
            state = np.asarray(ta.state, dtype=float)
            if not np.all(np.isfinite(state)):
                valid = False
                break
            if k < N:
                X_nodes[k - 1] = state[:7]
                lam_nodes[k] = state[7:14]
        if valid:
            return _ms_pack_z(lam_nodes, X_nodes)
        if fixed_lam0:
            raise RuntimeError('The supplied lam0_guess cannot generate a valid forward-forward node guess.')
    raise RuntimeError(f'Failed to generate a valid multiple-shooting guess after {attempts} attempts.')


In [18]:
def solve_multiple_shooting(N, z0=None, use_jac=True, eps=1e-05):
    """
    Solve the forward-forward multiple-shooting system.
    """
    if z0 is None:
        try:
            z0 = ms_build_initial_guess(N, lam0_guess=None)
        except RuntimeError:
            return None
    fun = lambda zz: ms_residual(zz, N)
    if use_jac:
        jac = lambda zz: ms_jacobian(zz, N)
        sol = root(fun, np.asarray(z0, dtype=float), jac=jac, method='hybr', tol=minpack_tol, options={'xtol': minpack_xtol, 'maxfev': minpack_maxfev, 'factor': minpack_factor, 'diag': [1.0] * (7 + 14 * (N - 1)), 'eps': minpack_eps})
    else:
        sol = root(fun, np.asarray(z0, dtype=float), method='hybr', tol=minpack_tol, options={'xtol': minpack_xtol, 'maxfev': minpack_maxfev, 'factor': minpack_factor, 'diag': [1.0] * (7 + 14 * (N - 1)), 'eps': minpack_eps})
    return sol


In [19]:
def fb_mid_build_initial_guess(m0=1.0, lam0_guess=None, rng=None):
    """
    Build z0 = [lam0(7), mf0(1), lam_rf0(3), lam_vf0(3)].

    When lam0_guess is supplied, the first seven entries are copied exactly.
    Extra terminal guesses are generated independently and therefore cannot
    alter the common lam0 sequence.
    """
    if rng is None:
        rng = np.random
    if lam0_guess is None:
        lam0 = 2.0 * rng.random(7) - 1.0
        lam0[6] = float(rng.random())
    else:
        lam0 = np.asarray(lam0_guess, dtype=float).copy()
        if lam0.shape != (7,):
            raise ValueError('lam0_guess must be a 7-dimensional vector.')
    mf0 = 0.9 * float(m0)
    lrf0 = 2.0 * rng.random(3) - 1.0
    lvf0 = 2.0 * rng.random(3) - 1.0
    return np.hstack([lam0, mf0, lrf0, lvf0])


In [20]:
def solve_fb_mid(z0=None, use_jac=True, use_SIMD=False):
    """
    Solve the midpoint forward-backward shooting system.
    """
    if z0 is None:
        z0 = fb_mid_build_initial_guess(m0=m0_nu)
    if not use_SIMD:
        if use_jac:
            r = root(lambda zz: fb_mid_residual(zz), x0=np.asarray(z0, float), jac=lambda zz: fb_mid_jacobian(zz), method='hybr', tol=minpack_tol, options={'xtol': minpack_xtol, 'maxfev': minpack_maxfev, 'factor': minpack_factor, 'diag': [1.0] * 7 + [1] + [1] * 3 + [1] * 3, 'eps': minpack_eps})
        else:
            r = root(lambda zz: fb_mid_residual(zz), x0=np.asarray(z0, float), method='hybr', tol=minpack_tol, options={'xtol': minpack_xtol, 'maxfev': minpack_maxfev, 'factor': minpack_factor, 'diag': [1.0] * 7 + [1] + [1] * 3 + [1] * 3, 'eps': minpack_eps})
    elif use_jac:
        r = root(lambda zz: fb_mid_residual_SIMD(zz), x0=np.asarray(z0, float), jac=lambda zz: fb_mid_jacobian_SIMD(zz), method='hybr', tol=minpack_tol, options={'xtol': minpack_xtol, 'maxfev': minpack_maxfev, 'factor': minpack_factor, 'diag': [1.0] * 7 + [1] + [1] * 3 + [1] * 3, 'eps': minpack_eps})
    else:
        r = root(lambda zz: fb_mid_residual_SIMD(zz), x0=np.asarray(z0, float), method='hybr', tol=minpack_tol, options={'xtol': minpack_xtol, 'maxfev': minpack_maxfev, 'factor': minpack_factor, 'diag': [1.0] * 7 + [1] + [1] * 3 + [1] * 3, 'eps': minpack_eps})
    return r


In [21]:
def fb_mid_residual(z, ratio=0.5):
    """
    Midpoint forward-backward shooting residual, with fixed parameter lambda0 = 1.
    Unknown vector: z = [lam0(7), mf(1), lrf(3), lvf(3)] ∈ R^14
    Residual: g(z) = X_L(t_mid) - X_R(t_mid) ∈ R^14
    """
    z = np.asarray(z, float).reshape(-1)
    lam0 = z[0:7]
    mf = float(z[7])
    lrf = z[8:11]
    lvf = z[11:14]
    t_mid = t0_nu + ratio * (tf_nu - t0_nu)
    ta.time = t0_nu
    ta.state[:7] = np.array(list(x0_nu) + [m0_nu], dtype=float)
    ta.state[7:14] = lam0.tolist()
    ta.propagate_until(t_mid)
    XL = np.array(ta.state[:14], dtype=float, copy=True)
    ta.time = tf_nu
    ta.state[0:6] = xf_nu
    ta.state[6] = mf
    ta.state[7:10] = lrf.tolist()
    ta.state[10:13] = lvf.tolist()
    ta.state[13] = 0.0
    ta.propagate_until(t_mid)
    XR = np.asarray(ta.state[:14], dtype=float)
    g_match = XL - XR
    return g_match


In [22]:
def fb_mid_jacobian(z, ratio=0.5):
    """
    Analytical Jacobian of the midpoint forward-backward shooting residual
    with fixed l0 = 1.

    ----------
    z : Unknown vector z = [lam0(7), mf(1), lrf(3), lvf(3)]
    ratio : Relative location of the matching point in [0, 1], defined by t_mid = t0_nu + ratio * (tf_nu - t0_nu). Default is 0.5.

    Returns
    -------
    J : ndarray, shape (14, 14)
        Analytical Jacobian of g(z) = X_L(t_mid) - X_R(t_mid)  with respect to z.
    """
    z = np.asarray(z, float).reshape(-1)
    lam0 = z[0:7]
    mf = float(z[7])
    lrf = z[8:11]
    lvf = z[11:14]
    l0 = lambda0
    t_mid = float(t0_nu) + ratio * (float(tf_nu) - float(t0_nu))
    X0_7 = np.array(list(x0_nu) + [m0_nu], dtype=float)
    X0L = np.r_[X0_7, lam0]
    _, _, PhiL = stm_segment_selected(X0L, l0, t0_nu, t_mid, active_cols=range(7, 14))
    dXL_dLam0 = PhiL[:, 7:14]
    Xf = np.zeros(14, dtype=float)
    Xf[0:6] = xf_nu
    Xf[6] = mf
    Xf[7:10] = lrf
    Xf[10:13] = lvf
    Xf[13] = 0.0
    _, _, PhiR = stm_segment_selected(Xf, l0, tf_nu, t_mid, active_cols=range(6, 13))
    dXR_dmf = PhiR[:, 6:7]
    dXR_dlrf = PhiR[:, 7:10]
    dXR_dlvf = PhiR[:, 10:13]
    J = np.zeros((14, 14), dtype=float)
    J[0:14, 0:7] = dXL_dLam0
    J[0:14, 7:8] = -dXR_dmf
    J[0:14, 8:11] = -dXR_dlrf
    J[0:14, 11:14] = -dXR_dlvf
    return J


In [23]:
def fb_mid_residual_SIMD(z, ratio=0.5):
    """
    Midpoint forward-backward shooting residual, with fixed parameter lambda0 = 1.
    Single z (shape (14,)): [Λ0(7), mf(1), λr,f(3), λv,f(3)]
    Return g (shape (14,)): [XL(t_mid)-XR(t_mid) (14)]
    """
    z = np.asarray(z, float).reshape(-1)
    lam0 = z[0:7]
    mf = float(z[7])
    lxf = z[8:14]
    t_mid = t0_nu + ratio * (tf_nu - t0_nu)
    x0_col = np.asarray(x0_nu, dtype=float).reshape(6, 1)
    L0 = np.asarray(lam0, float).reshape(7, 1)
    XL0 = np.vstack([x0_col, [[m0_nu]], L0])
    xf_col = np.asarray(xf_nu, dtype=float).reshape(6, 1)
    Lf = np.vstack([np.asarray(lxf, float).reshape(6, 1), np.zeros((1, 1))])
    XR0 = np.vstack([xf_col, [[mf]], Lf])
    X0 = np.concatenate([XL0, XR0], axis=1)
    ta_b2.set_time([t0_nu, tf_nu])
    ta_b2.state[:] = X0
    ta_b2.propagate_until([t_mid, t_mid])
    XL = ta_b2.state[:14, 0].copy()
    XR = ta_b2.state[:14, 1].copy()
    g_match = XL - XR
    return g_match


In [24]:
def fb_mid_jacobian_SIMD(z, ratio=0.5):
    """Two SIMD lanes, seven tangent directions per arc."""
    z = np.asarray(z, dtype=float)
    left = np.r_[x0_nu, m0_nu, z[:7]]
    right = np.r_[xf_nu, z[7], z[8:14], 0.]
    t_mid = float(t0_nu) + ratio * (float(tf_nu) - float(t0_nu))
    ta_var_fb_b2.set_time([t0_nu, tf_nu])
    ta_var_fb_b2.state[:14, :] = np.column_stack([left, right])
    initial = np.zeros((14, 7, 2))
    initial[np.arange(7, 14), np.arange(7), 0] = 1.
    initial[np.arange(6, 13), np.arange(7), 1] = 1.
    ta_var_fb_b2.state[14:, :] = initial.reshape(98, 2)
    ta_var_fb_b2.propagate_until([t_mid, t_mid])
    phi = ta_var_fb_b2.state[14:, :].reshape(14, 7, 2)
    return np.column_stack([phi[:, :, 0], -phi[:, :, 1]])


## Single shooting, FD


In [25]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
epsf = 1.0e-5
# epsf = 1.0e-3
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [26]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    time_start = time.perf_counter()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        # Call one more time with out = True to output terminal mass
        shooting(r['x'], epsf, lambda0, out=True, Time_out=None)
        # Save the global optimal solution
        final_mass = ta.state[6]*MUnit
        if final_mass>global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)


records_by_method['Single_FD'] = records


counter: 1/14 nfev: 127

  Final Mass: 2718.2885919564947


counter: 2/36 nfev: 296

  Final Mass: 2718.288591952573


counter: 3/43 nfev: 129

  Final Mass: 2718.288591955572


... (873 outputs left out)

  Final Mass: 2718.288591958256


counter: 444/4981 nfev: 337

  Final Mass: 2718.288591956549


Succeed to solve the problem!
Solved: 444 in 5000 (0.0888), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 96
Average number of function calls for convergent solution: 171
global_lambda: [-0.3472098485679188 -0.0574825063466253  0.1683148288139363
 -0.221195805748356  -0.5423588777416889  0.0003776045221329
  0.3901673099509018]


In [27]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_FD']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [28]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()


    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


34.8 s ± 1.2 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()


    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


6.37 s ± 144 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()


    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


27.4 s ± 270 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_FD'] = dict(
    method='Single shooting, FD', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_FD'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_FD']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Single shooting, AJ


In [25]:
# =========================
# Initialize solving time and global optimal solution
# =========================
epsf = 1.0e-5
# epsf = 1.0e-3
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0


# =========================
# Random initial guess loop
# =========================


In [26]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    time_start = time.perf_counter()

    r = root(
        lambda lam: shooting(lam, epsf, lambda0, False, None),
        x0 = x0,
        jac = lambda lam: shooting_jac(lam, epsf, lambda0),
        method = 'hybr',
        tol = minpack_tol,
        options = {
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        },
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])

    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        # Call one more time with out = True to output terminal mass
        shooting(r['x'], epsf, lambda0, out = True, Time_out = None)
        # Save the global optimal solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
records_by_method['Single_AJ'] = records


counter: 1/3 nfev: 67

  Final Mass: 2718.288591956366


counter: 2/9 nfev: 76

  Final Mass: 2718.288591956877


counter: 3/50 nfev: 153

  Final Mass: 2718.288591955624


... (1,102 outputs left out)

  Final Mass: 2718.288591958521


counter: 558/4987 nfev: 47

  Final Mass: 2718.2885919606547


Succeed to solve the problem!
Solved: 558 in 5000 (0.1116), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 49
Average number of function calls for convergent solution: 90
global_lambda: [-0.3472098485676602 -0.0574825063437643  0.168314828813562
 -0.2211958057457157 -0.5423588777421957  0.000377604522083
  0.390167309951409 ]


In [27]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [28]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()


    r = root(
        lambda lam: shooting(lam, epsf, lambda0, False, None),
        x0 = x0,
        jac = lambda lam: shooting_jac(lam, epsf, lambda0),
        method = 'hybr',
        tol = minpack_tol,
        options = {
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        },
    )


46 s ± 454 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()


    r = root(
        lambda lam: shooting(lam, epsf, lambda0, False, None),
        x0 = x0,
        jac = lambda lam: shooting_jac(lam, epsf, lambda0),
        method = 'hybr',
        tol = minpack_tol,
        options = {
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        },
    )


8.25 s ± 102 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()


    r = root(
        lambda lam: shooting(lam, epsf, lambda0, False, None),
        x0 = x0,
        jac = lambda lam: shooting_jac(lam, epsf, lambda0),
        method = 'hybr',
        tol = minpack_tol,
        options = {
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        },
    )


37.4 s ± 232 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_AJ'] = dict(
    method='Single shooting, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Forward-forward, AJ


In [25]:
# =========================
# Benchmark: Forward-Forward Multiple Shooting
# =========================
use_Jacobian = True
N = 2

# =========================
# Initialize solving time and global optimal solution
# =========================
# epsf = 1.0e-5
# lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0
build_failed = 0
solver_runs = 0

global_z = None
global_lambda = None
global_mf = -1.0


# =========================
# Random initial guess loop
# =========================


In [26]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Build method-specific node guesses from the same lam0_pool[i].
    # Keep this setup outside the solver timing interval.
    try:
        z0_ms = ms_build_initial_guess(
            N,
            lam0_guess=lam0_pool[i],
        )
    except RuntimeError:
        build_failed += 1
        save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)
        continue

    solver_runs += 1
    time_start = time.perf_counter()

    r = solve_multiple_shooting(
        N,
        z0=z0_ms,
        use_jac=use_Jacobian,
        eps=epsf,
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start

    if r is None:
        save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)
        continue

    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])

    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        lam_nodes, X_nodes, l0 = _ms_unpack_z(r['x'], N)

        # Output final mass using the current single-shooting interface
        shooting(lam_nodes[0], epsf, lambda0, out = True, Time_out = None)

        # Save the best solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_z = r['x'].copy()
            global_lambda = lam_nodes[0].copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved / repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Initial-node construction failures: {build_failed}")
    print(f"Solver runs after node construction: {solver_runs}")
    if solver_runs > 0:
        print(f"Average number of function calls per solver run: {total_nfev / solver_runs:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev / count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print("global_z:", global_z)
records_by_method['FF_AJ'] = records


counter: 1/34 nfev: 129

  Final Mass: 2718.28859197098


counter: 2/49 nfev: 114

  Final Mass: 2718.2885919990617


counter: 3/75 nfev: 120

  Final Mass: 2718.2885919978607


... (417 outputs left out)

  Final Mass: 2718.2885920000062


counter: 213/4973 nfev: 153

  Final Mass: 2718.288592008872


Succeed to solve the problem!
Solved: 213 in 5000 (0.0426), terminal mass: 2718.2886kg
Initial-node construction failures: 0
Solver runs after node construction: 5000
Average number of function calls per solver run: 56
Average number of function calls for convergent solution: 145
global_lambda: [-0.3472098485827072 -0.0574825064029133  0.1683148288130305
 -0.2211958059639613 -0.5423588776088588  0.0003776045231562
  0.3901673099560821]
global_z: [-0.3472098485827072 -0.0574825064029133  0.1683148288130305
 -0.2211958059639613 -0.5423588776088588  0.0003776045231562
  0.3901673099560821 -0.1173415253438862 -0.0653977147019
  0.2617233622975647 -0.221142014498455  -0.5421635470607263
  0.0064115201483959  0.1990323666241022  1.4556403805749154
  0.0542183251796401 -0.2596540793359665  0.0112583188130131
  0.0143212559199969 23.432820661494087   0.8212478765952396]


In [27]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(50)
accepted = np.array([r['accepted'] for r in records_by_method['FF_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [28]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Build method-specific node guesses from the same lam0_pool[i].
    # Keep this setup outside the solver timing interval.
    try:
        z0_ms = ms_build_initial_guess(
            N,
            lam0_guess=lam0_pool[i],
        )
    except RuntimeError:
        continue


    r = solve_multiple_shooting(
        N,
        z0=z0_ms,
        use_jac=use_Jacobian,
        eps=epsf,
    )


17.8 s ± 172 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Build method-specific node guesses from the same lam0_pool[i].
    # Keep this setup outside the solver timing interval.
    try:
        z0_ms = ms_build_initial_guess(
            N,
            lam0_guess=lam0_pool[i],
        )
    except RuntimeError:
        continue


    r = solve_multiple_shooting(
        N,
        z0=z0_ms,
        use_jac=use_Jacobian,
        eps=epsf,
    )


1.25 s ± 16.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Build method-specific node guesses from the same lam0_pool[i].
    # Keep this setup outside the solver timing interval.
    try:
        z0_ms = ms_build_initial_guess(
            N,
            lam0_guess=lam0_pool[i],
        )
    except RuntimeError:
        continue


    r = solve_multiple_shooting(
        N,
        z0=z0_ms,
        use_jac=use_Jacobian,
        eps=epsf,
    )


16.8 s ± 154 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['FF_AJ'] = dict(
    method='Forward-forward, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['FF_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['FF_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Forward-backward, AJ


In [25]:
# =========================
# Benchmark: Midpoint Forward-Backward Shooting
# =========================
use_Jacobian = True

# =========================
# Initialize solving time and global optimal solution
# =========================
count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_z = None
global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [26]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    z0_fb = fb_z0_pool[i].copy()

    time_start = time.perf_counter()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r.nfev

    res = np.linalg.norm(r.fun)
    if r.success and res < minpack_error:
        count_solved += 1
        converge_nfev += r.nfev
        print(f"counter: {count_solved}/{i} nfev: {r.nfev}", end=' ', flush=True)

        lam0_sol = r.x[:7].copy()

        # Output final mass using the current single-shooting interface
        shooting(lam0_sol, eps=epsf, lambda0=lambda0, out=True)

        # Save the best solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_z = r.x.copy()
            global_lambda = lam0_sol.copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved / repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev / repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev / count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print("global_z:", global_z)
records_by_method['FB_AJ'] = records


counter: 1/9 nfev: 41 

  Final Mass: 2718.288592083315


counter: 2/12 nfev: 101 

  Final Mass: 2718.288592056879


counter: 3/14 nfev: 64 

  Final Mass: 2718.28859203862


... (2,116 outputs left out)

counter: 1087/4996 nfev: 93 

  Final Mass: 2718.2885920350514


counter: 1088/4998 nfev: 60 

  Final Mass: 2718.2885920451727
Succeed to solve the problem!
Solved: 1088 in 5000 (0.2176), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 84
Average number of function calls for convergent solution: 89
global_lambda: [-0.3472098485729953 -0.0574825063592409  0.1683148287939153
 -0.2211958056751132 -0.5423588777248465  0.0003776045221966
  0.3901673099512415]
global_z: [-0.3472098485729953 -0.0574825063592409  0.1683148287939153
 -0.2211958056751132 -0.5423588777248465  0.0003776045221966
  0.3901673099512415  0.679572147987204  -0.0294343811323282
 -0.0877562927956804  0.1725780075245216 -0.2182541274905821
 -0.5347334352803719  0.0027725990852678]


In [27]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['FB_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [28]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )


1min 6s ± 828 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )


14.6 s ± 108 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )


50.8 s ± 533 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['FB_AJ'] = dict(
    method='Forward-backward, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['FB_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['FB_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Forward-backward SIMD, AJ


In [25]:
# =========================
# Benchmark: Midpoint Forward-Backward Shooting
# =========================
use_Jacobian = True

# =========================
# Initialize solving time and global optimal solution
# =========================
count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_z = None
global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [26]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Exactly the same full z0 used by method4 for trial i.
    z0_fb = fb_z0_pool[i].copy()

    time_start = time.perf_counter()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r.nfev

    res = np.linalg.norm(r.fun)
    if r.success and res < minpack_error:
        count_solved += 1
        converge_nfev += r.nfev
        print(f"counter: {count_solved}/{i} nfev: {r.nfev}", end=' ', flush=True)

        lam0_sol = r.x[:7].copy()

        # Output final mass using the current single-shooting interface
        shooting(lam0_sol, eps=epsf, lambda0=lambda0, out=True)

        # Save the best solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_z = r.x.copy()
            global_lambda = lam0_sol.copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved / repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev / repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev / count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print("global_z:", global_z)
records_by_method['FB_SIMD_AJ'] = records


counter: 1/3 nfev: 85 

  Final Mass: 2718.2885920376725


counter: 2/9 nfev: 41 

  Final Mass: 2718.2885920830195


counter: 3/11 nfev: 112 

  Final Mass: 2718.288592025232


... (2,078 outputs left out)

counter: 1080/4994 nfev: 61 

  Final Mass: 2718.2885920642884


counter: 1081/4998 nfev: 60 

  Final Mass: 2718.2885920477424
Succeed to solve the problem!
Solved: 1081 in 5000 (0.2162), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 84
Average number of function calls for convergent solution: 89
global_lambda: [-0.3472098485878886 -0.057482506356821   0.1683148288059724
 -0.2211958056914415 -0.5423588777267364  0.0003776045227686
  0.3901673099490645]
global_z: [-0.3472098485878886 -0.057482506356821   0.1683148288059724
 -0.2211958056914415 -0.5423588777267364  0.0003776045227686
  0.3901673099490645  0.6795721479897872 -0.029434381134378
 -0.0877562928016738  0.1725780075354515 -0.2182541275064985
 -0.5347334352821241  0.0027725990834596]


In [27]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['FB_SIMD_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [28]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Exactly the same full z0 used by method4 for trial i.
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )


53.2 s ± 521 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Exactly the same full z0 used by method4 for trial i.
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )


10.8 s ± 56 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Exactly the same full z0 used by method4 for trial i.
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )


41.6 s ± 364 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['FB_SIMD_AJ'] = dict(
    method='Forward-backward SIMD, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['FB_SIMD_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['FB_SIMD_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Forward-backward, FD


In [25]:
# =========================
# Benchmark: Midpoint Forward-Backward Shooting
# =========================
use_Jacobian = False

# =========================
# Initialize solving time and global optimal solution
# =========================
count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_z = None
global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [26]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    z0_fb = fb_z0_pool[i].copy()

    time_start = time.perf_counter()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r.nfev

    res = np.linalg.norm(r.fun)
    if r.success and res < minpack_error:
        count_solved += 1
        converge_nfev += r.nfev
        print(f"counter: {count_solved}/{i} nfev: {r.nfev}", end=' ', flush=True)

        lam0_sol = r.x[:7].copy()

        # Output final mass using the current single-shooting interface
        shooting(lam0_sol, eps=epsf, lambda0=lambda0, out=True)

        # Save the best solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_z = r.x.copy()
            global_lambda = lam0_sol.copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved / repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev / repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev / count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print("global_z:", global_z)
records_by_method['FB_FD'] = records


counter: 1/2 nfev: 177 

  Final Mass: 2718.288592047857


counter: 2/9 nfev: 287 

  Final Mass: 2718.288592048871


counter: 3/15 nfev: 257 

  Final Mass: 2718.288592047366


... (2,136 outputs left out)

  Final Mass: 2718.28859205213
counter: 1098/4995 nfev: 245 

  Final Mass: 2718.2885920613676


counter: 1099/4998 nfev: 175 

  Final Mass: 2718.2885920428103
Succeed to solve the problem!
Solved: 1099 in 5000 (0.2198), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 266
Average number of function calls for convergent solution: 255
global_lambda: [-0.3472098485812105 -0.0574825063565803  0.1683148288063663
 -0.2211958056813614 -0.5423588777251341  0.0003776045224858
  0.3901673099542219]
global_z: [-0.3472098485812105 -0.0574825063565803  0.1683148288063663
 -0.2211958056813614 -0.5423588777251341  0.0003776045224858
  0.3901673099542219  0.6795721479880247 -0.0294343811395556
 -0.0877562927980793  0.1725780075528654 -0.2182541274966495
 -0.5347334352806686  0.0027725990815759]


In [27]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['FB_FD']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [28]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )


1min 22s ± 1.21 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )


23.3 s ± 112 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )


57.8 s ± 372 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['FB_FD'] = dict(
    method='Forward-backward, FD', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['FB_FD'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['FB_FD']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Forward-backward SIMD, FD


In [25]:
# =========================
# Benchmark: Midpoint Forward-Backward Shooting
# =========================
use_Jacobian = False

# =========================
# Initialize solving time and global optimal solution
# =========================
count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_z = None
global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [26]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Exactly the same full z0 used by method4 for trial i.
    z0_fb = fb_z0_pool[i].copy()

    time_start = time.perf_counter()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r.nfev

    res = np.linalg.norm(r.fun)
    if r.success and res < minpack_error:
        count_solved += 1
        converge_nfev += r.nfev
        print(f"counter: {count_solved}/{i} nfev: {r.nfev}", end=' ', flush=True)

        lam0_sol = r.x[:7].copy()

        # Output final mass using the current single-shooting interface
        shooting(lam0_sol, eps=epsf, lambda0=lambda0, out=True)

        # Save the best solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_z = r.x.copy()
            global_lambda = lam0_sol.copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved / repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev / repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev / count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print("global_z:", global_z)
records_by_method['FB_SIMD_FD'] = records


counter: 1/3 nfev: 341 

  Final Mass: 2718.288592016381


counter: 2/19 nfev: 510 

  Final Mass: 2718.288592050005


counter: 3/32 nfev: 451 

  Final Mass: 2718.2885920393633


... (2,060 outputs left out)

counter: 1065/4992 nfev: 326 

  Final Mass: 2718.2885920443227


counter: 1066/4999 nfev: 376 

  Final Mass: 2718.2885920528815
Succeed to solve the problem!
Solved: 1066 in 5000 (0.2132), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 267
Average number of function calls for convergent solution: 263
global_lambda: [-0.3472098485592412 -0.0574825063537733  0.1683148287900852
 -0.2211958056706884 -0.542358877728019   0.0003776045217248
  0.3901673099461893]
global_z: [-0.3472098485592412 -0.0574825063537733  0.1683148287900852
 -0.2211958056706884 -0.542358877728019   0.0003776045217248
  0.3901673099461893  0.6795721479890535 -0.0294343811236863
 -0.0877562927942583  0.1725780074907084 -0.2182541274859288
 -0.5347334352836149  0.0027725990892293]


In [27]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['FB_SIMD_FD']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [28]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Exactly the same full z0 used by method4 for trial i.
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )


1min 8s ± 695 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Exactly the same full z0 used by method4 for trial i.
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )


14.9 s ± 145 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Exactly the same full z0 used by method4 for trial i.
    z0_fb = fb_z0_pool[i].copy()


    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )


53.2 s ± 219 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['FB_SIMD_FD'] = dict(
    method='Forward-backward SIMD, FD', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['FB_SIMD_FD'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['FB_SIMD_FD']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Results


In [4]:
headings = ['Method', 'Expected first solution [ms]', 'Successful mean [ms]',
            'Failed mean [ms]', 'All-trial mean [ms]', 'Converged', 'NFEV, successful']
table = ['| ' + ' | '.join(headings) + ' |', '| ' + ' | '.join(['---'] * len(headings)) + ' |']
for row in results_by_method.values():
    values = [row['method']]
    values += [f"{row[key]:.3f}" if np.isfinite(row[key]) else '—'
               for key in ('first_ms', 'successful_ms', 'failed_ms', 'all_ms')]
    values += [f"{row['accepted']}/{row['trials']}", f"{row['nfev_successful']:.1f}"]
    table.append('| ' + ' | '.join(values) + ' |')
display(Markdown('\n'.join(table)))


| Method | Expected first solution [ms] | Successful mean [ms] | Failed mean [ms] | All-trial mean [ms] | Converged | NFEV, successful |
| --- | --- | --- | --- | --- | --- | --- |
| Single shooting, FD | 784.615 | 135.582 | 60.534 | 69.674 | 444/5000 | 170.9 |
| Single shooting, AJ | 824.097 | 179.278 | 82.456 | 91.969 | 558/5000 | 89.9 |
| Forward-forward, AJ | 8343.296 | 627.437 | 350.136 | 355.424 | 213/5000 | 144.6 |
| Forward-backward, AJ | 611.634 | 144.533 | 127.286 | 133.091 | 1088/5000 | 89.3 |
| Forward-backward SIMD, AJ | 492.288 | 106.585 | 104.350 | 106.433 | 1081/5000 | 89.4 |
| Forward-backward, FD | 753.222 | 182.126 | 155.371 | 165.558 | 1099/5000 | 255.1 |
| Forward-backward SIMD, FD | 641.469 | 141.981 | 134.757 | 136.761 | 1066/5000 | 262.7 |